In [30]:
import csv
import os
import io
import base64
import PIL.Image
import PySimpleGUI as sg
import pandas as pd
import numpy as np

In [4]:
img_width = 750

def load_image(file, rotate=None):
   img = PIL.Image.open(file)
   new_size = (img_width, int(img_width * (img.size[1] / img.size[0])))

   img = img.resize(new_size)

   print(rotate)
   if (rotate != None):
      img = img.rotate(rotate, expand=True)

   with io.BytesIO() as bio:
      img.save(bio, format="PNG")
      del img
      return sg.Image(data=bio.getvalue())

In [50]:
def load_data():
  samples = pd.read_csv("data/dataset/data.csv", sep=";")
  return samples

# Selects a subset of the data for manual labelling
def select_validation_set(size=50):
  samples = load_data()
  val_set = samples.sample(n=size)
  val_set.to_csv("validation_set.csv", index=False)
  return val_set

# Create a csv file for storing labels
def prepare_label_csv():
  with open("labels.csv", "w", newline="") as f:
      writer = csv.writer(f)
      writer.writerow(["Num", "Filename", "Spelling", "Effort", "Layout", "Letter Formation", "Alteration"])


In [51]:
select_validation_set()
prepare_label_csv()

In [52]:
file_exists = os.path.isfile("labels.csv")
if not file_exists:
    raise Exception("Label CSV doesn't exist. Create first!")

samples = pd.read_csv("validation_set.csv")
print(samples)

def question(i, name, desc, lLabel = None, rLabel = None, min = 1, max = 5):
    return [
        sg.Column([
            sg.vtop([
                sg.Column([[sg.Text(name, font='20')]]),
                sg.Column([
                    [sg.Text(desc, font="12")],
                    sg.vbottom([
                        sg.Text(lLabel, visible=True if lLabel else False),
                        sg.Slider((min, max), expand_x=True, orientation='horizontal', key=("-" + name.upper().replace(" ", "-") + "-", i)),
                        sg.Text(rLabel, visible=True if rLabel else False),
                    ])
                ], expand_x=True)
            ])
        ], expand_x=True)
    ]

def make_view(image_filename, i, rotate=None):
    return [
        sg.pin(sg.Column(
            [sg.vtop([
                sg.Frame("Document", [
                    [sg.Button("Rotate (L)", key="-RotateCCW-"), sg.Button("Rotate (R)", key="-RotateCW-")],
                    [sg.Sizer(0, 900), load_image(image_filename, rotate)],
                ], key="-Document-"),
                sg.Column([
                    [sg.Frame("Questionnaire", [
                        question(i, "Spelling", "Are there comparatively many spelling errors?", "Yes", "No", 1, 0),
                        [sg.HorizontalSeparator()],
                        [sg.Text("For the first three questions, consider your overall impression of the document.")],
                        question(i, "Legibility", "Overall, how legible is the text when first reading?", "All w. legible", "Few w. legible"),
                        question(i, "Effort", "How much effort is required for you to read the document overall?", "No effort", "Extreme effort"),
                        question(i, "Layout", "An overall impression of the layout of writing\non the page. Well organised handwriting is consistent, with elements\nappropriately positioned in relation to each other (e.g. the position\nof the margin, placement of letters on the baseline, spaces within\nand between words).", "Good layout", "Bad layout"),
                        [sg.HorizontalSeparator()],
                        [sg.Text("Now focus on individual letters / words in more detail.")],
                        question(i, "Letter Formation", "An overall impression of letter formation.\nWell formed letters are appropriately shaped, contain all necessary\nelements, neat letter closures and are consistent in size and slope.", "All well-formed", "Most poorly formed"),
                        question(i, "Alteration", "An overall impression of the attempts made to rectify\nletters within words. Includes the addition of elements, re-tracing\n or re-writing of letters.", "None", "Most words"),
                        [sg.VPush()],
                    ], key="-Questionnaire-")],
                    [sg.VPush()],
                    [sg.Button("Next", key=("-Next-", i))]
                ], key="-ContentCol-")
            ])],
            k=("-ROW-", i)
        ))
    ]

start_index = 0

if os.path.isfile("labels.csv"):
    labels = pd.read_csv("labels.csv", sep=",")

    if labels.shape[0] > 0:
        start_index = labels.shape[0]

container = sg.Column([
    make_view(samples.iloc[start_index][1], samples.iloc[start_index][0])
], k="-CONTAINER-")

layout = [[container]]

window = sg.Window("Data Entry App", layout)

window.metadata = { "i": start_index, "rotations": [0]*samples.shape[0] }

while True:
    event, values = window.read()
    print(event)

    i = window.metadata["i"]
    sample = samples.iloc[i]

    if event == sg.WIN_CLOSED or event[0] == "-Exit-":
        break
    if "-RotateCCW-" in event:
        window.metadata["rotations"][i] += 90

        window.extend_layout(
            window["-CONTAINER-"],
            [make_view(sample[1], sample[0], 90)]
        )
        window[('-ROW-', sample["Num"])].update(visible=False)

    if "-RotateCW-" in event:
        window.metadata["rotations"][i] -= 90

        window.extend_layout(
            window["-CONTAINER-"],
            [make_view(sample[1], sample[0], window.metadata["rotations"][i])]
        )
        window[('-ROW-', sample["Num"])].update(visible=False)

    if event[0] == "-Next-":
        row = [
            sample["Num"],
            sample["Filename"],
            values[("-SPELLING-", sample["Num"])],
            values[("-EFFORT-", sample["Num"])],
            values[("-LAYOUT-", sample["Num"])],
            values[("-LETTER-FORMATION-", sample["Num"])],
            values[("-ALTERATION-", sample["Num"])],
        ]

        with open("labels.csv", "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(row)

        next_index = i + 1  # TODO: constrain next_index by number of samples

        window.extend_layout(
            window["-CONTAINER-"],
            [make_view(samples.iloc[next_index][1], samples.iloc[next_index]["Num"])]
        )
        window[('-ROW-', sample["Num"])].update(visible=False)
        window.metadata["i"] += 1


window.close()

     Num                     Filename
0    412  data/dataset/eng_EU_229.jpg
1   1297     data/dataset/f04-100.png
2    603  data/dataset/eng_NA_118.jpg
3    204  data/dataset/eng_EU_015.jpg
4   2015     data/dataset/n04-190.png
5    861     data/dataset/b03-114.png
6   1757     data/dataset/k04-093.png
7    970    data/dataset/c03-007b.png
8   2040     data/dataset/p01-147.png
9      0  data/dataset/eng_AF_001.jpg
10   530  data/dataset/eng_NA_038.jpg
11   388  data/dataset/eng_EU_204.jpg
12   111  data/dataset/eng_AS_009.jpg
13  1059     data/dataset/c06-076.png
14  1020     data/dataset/c04-017.png
15   550  data/dataset/eng_NA_059.jpg
16    64  data/dataset/eng_AF_084.jpg
17  2131     data/dataset/r03-084.png
18  2000     data/dataset/n04-060.png
19   531  data/dataset/eng_NA_039.jpg
20   188  data/dataset/eng_AS_097.jpg
21  1763     data/dataset/k04-119.png
22  1826     data/dataset/l04-113.png
23   362  data/dataset/eng_EU_178.jpg
24   200  data/dataset/eng_EU_010.jpg
25   599  da

/var/folders/7d/4wjlx8xx0hl76llb4wc5h09r0000gn/T/ipykernel_91383/2600407222.py:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  make_view(samples.iloc[start_index][1], samples.iloc[start_index][0])


('-Next-', 412)
None


/var/folders/7d/4wjlx8xx0hl76llb4wc5h09r0000gn/T/ipykernel_91383/2600407222.py:119: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  [make_view(samples.iloc[next_index][1], samples.iloc[next_index]["Num"])]


('-Next-', 1297)
None
('-Next-', 603)
None
('-Next-', 204)
None
**** EXITING ****


NameError: name 'exit' is not defined